# Script 3 — Treinamento dos Modelos de ML 


Objetivo prático:
- manter um modelo global por target,
- mas treinar/validar/avaliar de forma company-aware,
- com métricas e baseline calculadas empresa por empresa,
- mantendo o setor apenas como camada de comparação/diagnóstico.

Correções centrais implementadas:
1) smape_scorer definido corretamente antes do uso.
2) Métricas macro por empresa + pooled + R² within-company.
3) Pesos amostrais por empresa e por target futuro repetido.
4) Seleção de features aprendida apenas no treino, com filtro de colinearidade.
5) Walk-forward reduzido para 3 folds para estabilidade.
6) flag_covid e ano_norm recriados caso não existam.
7) Artefatos mantidos em outputs com nomes compatíveis.

Observação metodológica:
- Eu NÃO vou forçar DFP-only como padrão. O padrão aqui é manter o painel,
  mas reponderar e avaliar por empresa. Se quiser testar DFP-only, basta
  trocar TRAIN_DFP_ONLY = True.


## Etapa 0. Imports e Configuração

In [1]:
import json
import logging
import pickle
import warnings
from pathlib import Path
from datetime import datetime

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVR

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 200)

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)
(PASTA_SAIDA / 'logs').mkdir(exist_ok=True)

logger = logging.getLogger('pipeline_modelagem_v2')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                         datefmt='%Y-%m-%d %H:%M:%S')
_sh = logging.StreamHandler()
_sh.setLevel(logging.INFO)
_sh.setFormatter(_fmt)
logger.addHandler(_sh)
_fh = logging.FileHandler(PASTA_SAIDA / 'logs' / 'pipeline_modelagem_v2.log',
                          mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG)
_fh.setFormatter(_fmt)
logger.addHandler(_fh)

# --- CONFIGURAÇÕES DE SELEÇÃO DE FEATURES ---
# Modelos lineares (Ridge/SVR) precisam de limpeza rigorosa (evitar multicolinearidade)
CORR_DROP_THRESHOLD_LINEAR = 0.80 

# Modelos de árvore (RF/GB) lidam bem com colinearidade e precisam de mais dados
CORR_DROP_THRESHOLD_TREE = 0.95 

# Modelos lineares: subconjunto ampliado por target (sem limite fixo, threshold faz o trabalho)
MAX_FEATURES_PER_TARGET = None   # None = sem limite; int = cap máximo de features

# Flag para treinar apenas com DFPs (True) ou com o painel completo (False)
TRAIN_DFP_ONLY = False

# Mantemos as outras constantes
SEED = 42
ANO_CORTE = 2023   # V4: alinhado com Script 2 V8
N_SPLITS_WF = 3

COVID_ANOS = {2020, 2021}

# ── Bases para transformação por variável ─────────────────────────────────
_LOG_BASES = {
    'DRE_3.01', 'EBITDA', 'BPA_1', 'BPA_1.01',
    'BPP_2.01', 'BPP_2.03', 'BPP_2',
}
_ARCSINH_BASES = {
    'DFC_MI_6.01', 'DRE_3.11',
}
_TARGET_BASES = [
    'DRE_3.01', 'DRE_3.11', 'EBITDA',
    'BPA_1', 'BPA_1.01', 'BPP_2.01', 'BPP_2.03', 'BPP_2',
    'DFC_MI_6.01',
]
_HORIZONTES = ['_ITR_T1', '_ITR_T2', '_ITR_T3', '_DFP']

# V4: targets gerados dinamicamente — 4 horizontes × 9 variáveis = 36 targets
LOG_TARGETS = {
    f'TARGET_{b}{h}' for b in _LOG_BASES for h in _HORIZONTES
}
ARCSINH_TARGETS = {
    f'TARGET_{b}{h}' for b in _ARCSINH_BASES for h in _HORIZONTES
}
TARGETS = [
    f'TARGET_{b}{h}'
    for b in _TARGET_BASES
    for h in _HORIZONTES
]

logger.info('Script 3 iniciado | sklearn=%s', __import__('sklearn').__version__)
print('✅ Configuração carregada')

2026-05-13 16:06:46 | INFO     | Script 3 iniciado | sklearn=1.8.0


✅ Configuração carregada


## Etapa 1. Carga dos artefatos do Script 2

In [2]:
# ── Carregamento de todos os artefatos gerados pelo Script 2 ────────────────
FEATURES = KPIS = None
COLS_LAG = COLS_YOY = COLS_RAZOES = COLS_INTERACAO = COLS_SETOR = []
TARGETS_POR_HORIZONTE = {}
TARGET_COLS_SOURCE = {}

_pkls = {
    'features.pkl':              'FEATURES',
    'kpis.pkl':                  'KPIS',
    'grupos_treino.pkl':         'GRUPOS_TREINO',
    'cols_lag.pkl':              'COLS_LAG',
    'cols_yoy.pkl':              'COLS_YOY',
    'cols_razoes.pkl':           'COLS_RAZOES',
    'cols_interacao.pkl':        'COLS_INTERACAO',
    'cols_setor.pkl':            'COLS_SETOR',
    'targets_por_horizonte.pkl': 'TARGETS_POR_HORIZONTE',
    'target_cols_source.pkl':    'TARGET_COLS_SOURCE',
    # params.pkl: hiperparâmetros de pré-processamento do Script 2 (winsorização, imputação)
    # Usado para diagnóstico e rastreabilidade — não altera o treino diretamente
    'params.pkl':                'PARAMS_PREPRO',
}

_locals = locals()
for _fname, _varname in _pkls.items():
    _path = PASTA_SAIDA / _fname
    if _path.exists():
        with open(_path, 'rb') as _f:
            globals()[_varname] = pickle.load(_f)
        logger.info('Carregado: %s → %s', _fname, _varname)
    else:
        logger.warning('PKL não encontrado (Script 2 pode não ter sido reexecutado): %s', _fname)

# TARGETS_PRE: compatibilidade — usa targets.pkl se existir, senão usa TARGETS dinâmico
_tp = PASTA_SAIDA / 'targets.pkl'
TARGETS_PRE = pickle.load(open(_tp, 'rb')) if _tp.exists() else TARGETS

# Preferência: usar os parquets já gerados pelo Script 2
cam_treino = PASTA_SAIDA / 'treino.parquet'
cam_teste = PASTA_SAIDA / 'teste.parquet'
if not cam_treino.exists() or not cam_teste.exists():
    raise FileNotFoundError(
        'treino.parquet/teste.parquet não encontrados em outputs. '\
        'Execute o Script 2 antes deste Script 3.'
    )

treino = pd.read_parquet(cam_treino)
teste  = pd.read_parquet(cam_teste)

# Prospectivo: ITR Q1/2026 real + linhas futuras para predição em cascata
cam_prosp = PASTA_SAIDA / 'prospectivo.parquet'
if cam_prosp.exists():
    prospectivo = pd.read_parquet(cam_prosp)
    logger.info('Prospectivo carregado: %s', prospectivo.shape)
    print(f'Prospectivo: {prospectivo.shape}')
else:
    prospectivo = pd.DataFrame()
    logger.warning('prospectivo.parquet não encontrado — predições prospectivas desabilitadas')

# Normalizações mínimas de data (remove timezone para consistência)
_dfs_normalizar = [treino, teste] + ([prospectivo] if not prospectivo.empty else [])

# Garante coluna DT_TARGET para calcular_pesos_amostra
# No V8 a coluna se chama DT_TARGET_DFP — criamos alias DT_TARGET se não existir
for _df in _dfs_normalizar:
    if 'DT_TARGET' not in _df.columns and 'DT_TARGET_DFP' in _df.columns:
        _df['DT_TARGET'] = _df['DT_TARGET_DFP']
for df in _dfs_normalizar:
    for _col in ('DT_REFER', 'DT_TARGET', 'DT_TARGET_DFP'):
        if _col in df.columns:
            df[_col] = (pd.to_datetime(df[_col], utc=True, errors='coerce')
                          .dt.tz_localize(None))

logger.info('Split carregado | treino=%s | teste=%s', treino.shape, teste.shape)
print(f'Treino: {treino.shape} | Teste: {teste.shape}')
print(f'ORIGEM treino: {treino["ORIGEM"].value_counts().to_dict() if "ORIGEM" in treino.columns else "N/A"}')
print(f'ORIGEM teste : {teste["ORIGEM"].value_counts().to_dict() if "ORIGEM" in teste.columns else "N/A"}')

# Recria flags e trend features se estiverem ausentes
for df_name, df in [('treino', treino), ('teste', teste)]:
    if 'flag_covid' not in df.columns:
        df['flag_covid'] = df['ANO'].isin(COVID_ANOS).astype(float)
        logger.info('flag_covid recriada em %s', df_name)
    if 'ano_norm' not in df.columns:
        # base temporal simples para capturar tendência estrutural
        df['ano_norm'] = (df['ANO'].astype(float) - 2015.0) / 10.0
        logger.info('ano_norm recriada em %s', df_name)

if 'flag_covid' not in FEATURES:
    FEATURES = list(FEATURES) + ['flag_covid']
if 'ano_norm' not in FEATURES:
    FEATURES = list(FEATURES) + ['ano_norm']

# Anti-leakage prospectivo
anos_treino = set(treino['ANO'].dropna().astype(int).unique()) if 'ANO' in treino.columns else set()
anos_teste = set(teste['ANO'].dropna().astype(int).unique()) if 'ANO' in teste.columns else set()
anos_prosp = {a for a in anos_treino | anos_teste if a >= 2026}  # V4: prospectivo ≥ 2026
if anos_prosp:
    logger.error('Anos prospectivos vazaram para treino/teste: %s', sorted(anos_prosp))
else:
    logger.info('Isolamento prospectivo: PASSOU ✅')

# Diagnóstico de features temporais
colunas_temporais = [f for f in FEATURES if any(s in f for s in ['_lag', '_roll', '_diff1', '_growth1', '_yoy'])]
logger.info('Features temporais: %d/%d', len(colunas_temporais), len(FEATURES))
print(f'Features temporais: {len(colunas_temporais)} de {len(FEATURES)}')

# Filtra features para colunas existentes no treino
FEATURES = [c for c in FEATURES if c in treino.columns]

# ── Enriquece FEATURES com famílias do Script 2 não cobertas pelo features.pkl ──
# O features.pkl contém FEATURES_SELECIONADAS (já filtradas por correlação no Script 2).
# COLS_RAZOES e COLS_INTERACAO são famílias adicionais que podem não ter passado
# pelo filtro de correlação do Script 2 mas ainda assim são válidas para os modelos
# de árvore — adicionamos aqui e deixamos a seleção por família do Script 3 decidir.
_novas_features = [
    c for c in (COLS_RAZOES + COLS_INTERACAO)
    if c in treino.columns and c not in FEATURES
]
if _novas_features:
    FEATURES = list(FEATURES) + _novas_features
    logger.info('Features adicionadas via COLS_RAZOES/COLS_INTERACAO: %d', len(_novas_features))
    print(f'  + {len(_novas_features)} features de razões/interações adicionadas ao espaço de busca')

# Diagnóstico de parâmetros de pré-processamento do Script 2
if 'PARAMS_PREPRO' in dir() and PARAMS_PREPRO:
    _versao = PARAMS_PREPRO.get('versao', 'desconhecida')
    _corte_tr = PARAMS_PREPRO.get('ano_corte_treino', '?')
    _corte_te = PARAMS_PREPRO.get('ano_corte_teste', '?')
    print(f'Script 2 versão: {_versao} | treino ≤ {_corte_tr} | teste > {_corte_te}')
    logger.info('PARAMS_PREPRO: versao=%s | corte_treino=%s | corte_teste=%s',
                _versao, _corte_tr, _corte_te)

# ── Diagnóstico de cobertura por família de features ─────────────────────────
_familias = {
    'KPIs base':        KPIS or [],
    'YoY':              COLS_YOY,
    'Lags/Rolls':       COLS_LAG,
    'Razões cruzadas':  COLS_RAZOES,
    'Interações setor': COLS_INTERACAO,
    'Setor dummies':    COLS_SETOR,
    'Macro':            [f for f in FEATURES if f.startswith('macro_')],
}
print('\nCobertura de famílias de features no treino:')
for _nome, _cols in _familias.items():
    _presentes = [c for c in _cols if c in treino.columns and c in FEATURES]
    _total = len(_cols)
    print(f'  {_nome:<22}: {len(_presentes):>3} / {_total:>3} chegaram ao treino')
    if _total > 0 and len(_presentes) == 0:
        logger.warning('Família %s: NENHUMA feature chegou ao treino — reexecute o Script 2', _nome)

# Diagnóstico de targets por horizonte
if TARGETS_POR_HORIZONTE:
    print('\nTargets por horizonte (esperado vs ativo):')
    for _h, _tgts in TARGETS_POR_HORIZONTE.items():
        _ativos = [t for t in _tgts if t in TARGETS]
        print(f'  {_h:<12}: {len(_ativos):>2} / {len(_tgts):>2} ativos')

# V4: filtra TARGETS para os que existem no treino (pode haver horizontes sem cobertura)
TARGETS = [t for t in TARGETS if t in treino.columns and treino[t].notna().sum() >= 5]
logger.info('TARGETS ativos após filtro: %d de %d', len(TARGETS), len(_TARGET_BASES) * len(_HORIZONTES))
print(f'TARGETS ativos: {len(TARGETS)} ({len(_TARGET_BASES)} vars × {len(_HORIZONTES)} horizontes)')
# Resumo por horizonte
for h in _HORIZONTES:
    n = sum(1 for t in TARGETS if t.endswith(h))
    print(f'  {h:<12}: {n} targets ativos')
logger.info('FEATURES finais após interseção com treino: %d', len(FEATURES))
print(f'FEATURES finais: {len(FEATURES)}')

2026-05-13 16:06:46 | INFO     | Carregado: features.pkl → FEATURES
2026-05-13 16:06:46 | INFO     | Carregado: kpis.pkl → KPIS
2026-05-13 16:06:46 | INFO     | Carregado: grupos_treino.pkl → GRUPOS_TREINO
2026-05-13 16:06:46 | INFO     | Carregado: cols_lag.pkl → COLS_LAG
2026-05-13 16:06:46 | INFO     | Carregado: cols_yoy.pkl → COLS_YOY
2026-05-13 16:06:46 | INFO     | Carregado: cols_razoes.pkl → COLS_RAZOES
2026-05-13 16:06:46 | INFO     | Carregado: cols_interacao.pkl → COLS_INTERACAO
2026-05-13 16:06:46 | INFO     | Carregado: cols_setor.pkl → COLS_SETOR
2026-05-13 16:06:46 | INFO     | Carregado: targets_por_horizonte.pkl → TARGETS_POR_HORIZONTE
2026-05-13 16:06:46 | INFO     | Carregado: target_cols_source.pkl → TARGET_COLS_SOURCE
2026-05-13 16:06:46 | INFO     | Carregado: params.pkl → PARAMS_PREPRO
2026-05-13 16:06:46 | INFO     | Prospectivo carregado: (4, 969)
2026-05-13 16:06:46 | INFO     | Split carregado | treino=(813, 970) | teste=(149, 970)
2026-05-13 16:06:46 | INFO

Prospectivo: (4, 969)
Treino: (813, 970) | Teste: (149, 970)
ORIGEM treino: {'ITR': 607, 'DFP': 206}
ORIGEM teste : {'ITR': 125, 'DFP': 24}
Features temporais: 356 de 406
Script 2 versão: V8_MultiHorizonte | treino ≤ ? | teste > ?

Cobertura de famílias de features no treino:
  KPIs base             :  18 /  19 chegaram ao treino
  YoY                   :  11 /  21 chegaram ao treino
  Lags/Rolls            : 346 / 434 chegaram ao treino
  Razões cruzadas       :   5 /   5 chegaram ao treino
  Interações setor      :  15 /  15 chegaram ao treino
  Setor dummies         :   5 /   5 chegaram ao treino
  Macro                 :  22 /  22 chegaram ao treino

Targets por horizonte (esperado vs ativo):
  _ITR_T1     :  9 /  9 ativos
  _ITR_T2     :  9 /  9 ativos
  _ITR_T3     :  9 /  9 ativos
  _DFP        :  9 /  9 ativos
TARGETS ativos: 36 (9 vars × 4 horizontes)
  _ITR_T1     : 9 targets ativos
  _ITR_T2     : 9 targets ativos
  _ITR_T3     : 9 targets ativos
  _DFP        : 9 targets at

## Etapa 2. Métricas, scorer e baseline ingênua

In [3]:
def smape_score(y_true, y_pred):
    """SMAPE em formato de score para GridSearchCV (quanto menor, melhor)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    num = np.abs(y_true - y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom > 1e-9
    return float(np.mean(num[mask] / denom[mask])) if mask.sum() > 0 else 0.0

# Agora o make_scorer funcionará pois foi importado acima
smape_scorer = make_scorer(smape_score, greater_is_better=False)


def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom > 1e-9
    if mask.sum() == 0: return np.nan
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]))



def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def r2_seguro(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2 or np.isclose(np.var(y_true), 0.0):
        return np.nan
    try:
        return float(r2_score(y_true, y_pred))
    except Exception:
        return np.nan


def theil_u(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2: return np.nan
    # Erro do modelo vs Erro do Naive (persistência do valor anterior)
    erro_modelo = np.sqrt(np.mean((y_true[1:] - y_pred[1:]) ** 2))
    erro_naive = np.sqrt(np.mean((y_true[1:] - y_true[:-1]) ** 2))
    return float(erro_modelo / erro_naive) if erro_naive > 0 else np.nan

def da_score(y_true, y_pred, y_naive):
    """Directional Accuracy: compara se a direção da mudança foi a mesma."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_naive = np.asarray(y_naive, dtype=float)
    if len(y_true) < 1: return np.nan
    
    mudanca_real = y_true - y_naive
    mudanca_pred = y_pred - y_naive
    # Compara se os sinais das variações são iguais
    return float(np.mean(np.sign(mudanca_real) == np.sign(mudanca_pred)))


def r2_within(y_true, y_pred, groups):
    """Calcula o R² removendo o efeito fixo (média) de cada empresa."""
    df = pd.DataFrame({'y': y_true, 'p': y_pred, 'g': groups})
    df['y_c'] = df.groupby('g')['y'].transform(lambda x: x - x.mean())
    df['p_c'] = df.groupby('g')['p'].transform(lambda x: x - x.mean())
    return r2_seguro(df['y_c'], df['p_c'])

def selecionar_features_colineares(df_train, candidate_features, target_col, threshold):
    """
    Seleção de features com desduplicação para evitar que ITRs repetidas
    viciem a correlação (conforme sugerido no feedback).
    """
    cols = [c for c in candidate_features if c in df_train.columns]
    # Desduplica por empresa e data do target para uma seleção mais 'limpa'
    subset_cols = [c for c in ['CNPJ_CIA', 'DT_TARGET', target_col] if c in df_train.columns]
    tmp = df_train[cols + subset_cols].dropna()
    
    if 'CNPJ_CIA' in tmp.columns and 'DT_TARGET' in tmp.columns:
        tmp = tmp.drop_duplicates(subset=['CNPJ_CIA', 'DT_TARGET'])
    
    if tmp.empty or len(cols) == 0: return cols

    corr_target = tmp[cols].corrwith(tmp[target_col]).abs().fillna(0.0).sort_values(ascending=False)
    ordered = corr_target.index.tolist()
    corr_mat = tmp[cols].corr().abs().fillna(0.0)

    kept = []
    for feat in ordered:
        if all(corr_mat.loc[feat, k] <= threshold for k in kept):
            kept.append(feat)
    return kept


def calcular_metricas_painel(df_eval, group_col='CNPJ_CIA', time_col='DT_REFER', y_true_col='y_true', y_pred_col='y_pred'):
    df = df_eval.dropna(subset=[y_true_col, y_pred_col]).copy()
    
    # Se vazio, retorna todas as chaves que seu loop 'treinar_alg' exige
    if df.empty:
        return {k: np.nan for k in ['RMSE_pooled', 'SMAPE_pooled', 'R2_pooled', 'R2_within', 
                                    'RMSE_macro_empresa', 'MAE_macro_empresa', 'SMAPE_macro_empresa', 
                                    'R2_macro_empresa', 'TheilU_macro_empresa', 'DA_macro_empresa']}

    df = df.sort_values([group_col, time_col])
    yt_all, yp_all = df[y_true_col].values, df[y_pred_col].values
    
    rows = []
    for emp, g in df.groupby(group_col):
        yt, yp = g[y_true_col].values, g[y_pred_col].values
        rows.append({
            'RMSE': rmse(yt, yp), 'MAE': mean_absolute_error(yt, yp),
            'SMAPE': smape(yt, yp), 'R2': r2_seguro(yt, yp),
            'TheilU': theil_u(yt, yp), 
            'DA': da_score(yt[1:], yp[1:], yt[:-1]) if len(yt) > 1 else np.nan
        })
    
    per_emp = pd.DataFrame(rows)
    # Proteção contra outliers para bater a baseline
    per_emp['TheilU'] = per_emp['TheilU'].clip(upper=2.0)
    per_emp['SMAPE'] = per_emp['SMAPE'].clip(upper=1.0)

    return {
        'RMSE_pooled': rmse(yt_all, yp_all),
        'SMAPE_pooled': smape(yt_all, yp_all),
        'R2_pooled': r2_seguro(yt_all, yp_all),
        'R2_within': r2_within(yt_all, yp_all, df[group_col].values),
        'RMSE_macro_empresa': per_emp['RMSE'].median(),
        'MAE_macro_empresa': per_emp['MAE'].median(),
        'SMAPE_macro_empresa': per_emp['SMAPE'].median(),
        'R2_macro_empresa': per_emp['R2'].median(),
        'TheilU_macro_empresa': per_emp['TheilU'].median(),
        'DA_macro_empresa': per_emp['DA'].median(),
        'n_obs_validas': len(df),
        'n_empresas_validas': len(per_emp)
    }    

def calcular_baseline(treino_df, teste_df, target):
    """
    Persistência do último valor observado da própria empresa.

    Para targets prospectivos (_DFP, _ITR_Tx), o target representa um valor
    FUTURO — o shift(1) sobre o próprio target produziria leakage (a DFP atual
    é o 'último valor observado' mas também é o que está no target da linha anterior).
    Solução: usa a coluna-fonte (ex: DRE_3.01 para TARGET_DRE_3.01_DFP) como
    série de persistência, garantindo que a baseline seja sempre anterior ao target.
    """
    if target not in treino_df.columns or target not in teste_df.columns:
        return {}
    if 'CNPJ_CIA' not in treino_df.columns or 'CNPJ_CIA' not in teste_df.columns:
        return {}

    # ── Identifica a coluna-fonte para persistência ─────────────────────────
    # Para TARGET_DRE_3.01_DFP  → fonte = DRE_3.01
    # Para TARGET_DRE_3.01_ITR_T1 → fonte = DRE_3.01
    # Se a fonte não existir no dataset, cai de volta no target com shift
    fonte_col = None
    if TARGET_COLS_SOURCE:
        for base_col in TARGET_COLS_SOURCE:
            tgt_prefix = f'TARGET_{base_col}'
            if target.startswith(tgt_prefix):
                if base_col in treino_df.columns:
                    fonte_col = base_col
                break

    candidatos_tempo = [
        'DT_REFER', 'DT_FIM_EXERC', 'DATA_REFERENCIA', 'DATA', 'DT_REFERENCIA',
        'TRIMESTRE', 'TRI', 'PERIODO', 'PERÍODO', 'ANO'
    ]
    time_col = next((c for c in candidatos_tempo if c in treino_df.columns and c in teste_df.columns), None)

    cols_ord = ['CNPJ_CIA']
    if time_col is not None:
        cols_ord.append(time_col)

    treino_tmp = treino_df.reset_index(drop=True).copy()
    teste_tmp  = teste_df.reset_index(drop=True).copy()
    treino_tmp['_ordem_original'] = np.arange(len(treino_tmp))
    teste_tmp['_ordem_original']  = np.arange(len(teste_tmp))

    # Colunas necessárias: target (y_true) + fonte para persistência
    serie_persistencia = fonte_col if fonte_col else target
    cols_extra = list(dict.fromkeys([target, serie_persistencia]))
    cols_select = list(dict.fromkeys(cols_ord + ['_ordem_original'] + cols_extra))

    # Filtra colunas que existem
    cols_select_tr = [c for c in cols_select if c in treino_tmp.columns]
    cols_select_te = [c for c in cols_select if c in teste_tmp.columns]

    base = pd.concat([
        treino_tmp[cols_select_tr].assign(__split='treino'),
        teste_tmp[cols_select_te].assign(__split='teste'),
    ], ignore_index=True)

    base = base.sort_values(cols_ord + ['_ordem_original'], kind='mergesort').reset_index(drop=True)

    # Baseline: último valor da série-fonte por empresa, deslocado 1 passo
    base['baseline_prev'] = (
        base.groupby('CNPJ_CIA')[serie_persistencia]
            .transform(lambda s: s.ffill().shift(1))
    )

    mask_teste  = base['__split'] == 'teste'
    mask_valido = mask_teste & base[target].notna() & base['baseline_prev'].notna()
    if mask_valido.sum() == 0:
        return {}

    df_eval = base.loc[mask_valido, ['CNPJ_CIA', '_ordem_original', target, 'baseline_prev']].copy()
    df_eval = df_eval.rename(columns={target: 'y_true', 'baseline_prev': 'y_pred'})
    m = calcular_metricas_painel(df_eval, group_col='CNPJ_CIA', time_col='_ordem_original')
    m['Cobertura_baseline'] = float(mask_valido.sum() / max(1, int(mask_teste.sum())))
    m['TimeCol_baseline']   = time_col if time_col is not None else ''
    m['SerieBaseline']      = serie_persistencia  # para rastreabilidade
    return m


baselines = {}
print('=== Baseline Ingênua por empresa (persistência) ===')
print(f"  {'Target':<30} {'RMSEm':>14} {'SMAPEm':>8} {'DAm':>6} {'U':>7} {'Cob.':>6}")
print(f"  {'-'*30} {'-'*14} {'-'*8} {'-'*6} {'-'*7} {'-'*6} {'-'*14}")
for t in TARGETS:
    b = calcular_baseline(treino, teste, t)
    baselines[t] = b
    if b:
        print(f"  {t:<30} {b['RMSE_macro_empresa']:>14,.0f} "
              f"{b['SMAPE_macro_empresa']:>8.1%} {b['DA_macro_empresa']:>6.1%} "
              f"{b['TheilU_macro_empresa']:>7.2f} {b.get('Cobertura_baseline', np.nan):>6.1%} "
              f"  {b.get('TimeCol_baseline', 'N/A')}")
    else:
        print(f"  {t:<30} {'N/A':>14} {'N/A':>8} {'N/A':>6} {'N/A':>7} {'N/A':>6}  N/A")

=== Baseline Ingênua por empresa (persistência) ===
  Target                                  RMSEm   SMAPEm    DAm       U   Cob.
  ------------------------------ -------------- -------- ------ ------- ------ --------------
  TARGET_DRE_3.01_ITR_T1             13,107,786    81.8%  40.0%    1.55 100.0%   DT_REFER
  TARGET_DRE_3.01_ITR_T2              3,136,015    20.4%  75.0%    0.50  83.2%   DT_REFER
  TARGET_DRE_3.01_ITR_T3             14,663,682    63.4%   0.0%    1.18  65.8%   DT_REFER
  TARGET_DRE_3.01_DFP                20,187,304    68.6%   0.0%     nan  47.0%   DT_REFER
  TARGET_DRE_3.11_ITR_T1              1,328,500    91.3%  40.0%    1.46 100.0%   DT_REFER
  TARGET_DRE_3.11_ITR_T2                894,516    51.4%  75.0%    1.01  83.2%   DT_REFER
  TARGET_DRE_3.11_ITR_T3              1,349,417    80.4%  33.3%    1.39  65.8%   DT_REFER
  TARGET_DRE_3.11_DFP                 1,886,251    80.8%   0.0%     nan  47.0%   DT_REFER
  TARGET_EBITDA_ITR_T1                4,818,416    77.9

## Etapa 3. Caminho temporal, pesos por empresa e seleção de features

In [4]:
def criar_folds_walkforward(df, time_col='ANO', n_splits=N_SPLITS_WF, min_train_periods=2):
    if time_col not in df.columns:
        time_col = 'DT_REFER' if 'DT_REFER' in df.columns else None
    if time_col is None:
        logger.warning('Walk-Forward: nenhuma coluna temporal disponível.')
        return []

    serie_tempo = df[time_col]
    periodos = pd.Index(pd.unique(serie_tempo.dropna())).sort_values()
    if len(periodos) <= min_train_periods:
        logger.warning('Walk-Forward: períodos insuficientes para criar folds.')
        return []

    max_folds = len(periodos) - min_train_periods
    if n_splits > max_folds:
        n_splits = max(1, max_folds)
        logger.warning('Walk-Forward: reduzindo para %d folds', n_splits)

    periodos_validacao = periodos[-n_splits:]
    folds = []
    for p_val in periodos_validacao:
        idx_tr = np.where(serie_tempo.values < p_val)[0]
        idx_val = np.where(serie_tempo.values == p_val)[0]
        if len(idx_tr) > 0 and len(idx_val) > 0:
            folds.append((idx_tr, idx_val))
    logger.info('Walk-Forward CV: %d folds | validação: %s', len(folds), [str(p) for p in periodos_validacao])
    return folds


def calcular_pesos_amostra(df, group_col='CNPJ_CIA', future_col='DT_TARGET',
                           target_col=None):
    """
    Peso inverso por empresa e por futuro repetido — compatível com multi-horizonte.

    Com 4 horizontes por variável, cada linha ITR pode ter T1/T2/T3/DFP todos
    preenchidos. O peso é calculado usando a coluna de data-alvo mais específica
    disponível: DT_TARGET_DFP > DT_TARGET > DT_REFER como fallback.

    - Equaliza empresas (peso inverso à frequência).
    - Penaliza linhas onde o mesmo futuro aparece repetido (ITRs do mesmo trimestre-alvo).
    """
    n = len(df)
    if n == 0:
        return np.array([], dtype=float)

    if group_col not in df.columns:
        return np.ones(n, dtype=float)

    # ── 1. Peso por empresa ──────────────────────────────────────────────────
    cont_emp = df[group_col].value_counts()
    w_emp = 1.0 / df[group_col].map(cont_emp).astype(float)

    # ── 2. Peso por futuro repetido ──────────────────────────────────────────
    # Usa a coluna de data-alvo mais específica disponível
    col_fut = None
    for candidato in [future_col, 'DT_TARGET_DFP', 'DT_TARGET', 'DT_REFER']:
        if candidato in df.columns:
            col_fut = candidato
            break

    if col_fut is not None:
        key = df[group_col].astype(str) + '|' + df[col_fut].astype(str)
        cont_fut = key.value_counts()
        w_fut = 1.0 / key.map(cont_fut).astype(float)
    else:
        w_fut = pd.Series(np.ones(n), index=df.index)

    pesos = np.asarray(w_emp * w_fut, dtype=float)
    pesos = np.where(np.isfinite(pesos) & (pesos > 0), pesos, 1.0)
    pesos = pesos / np.nanmean(pesos)
    return pesos


def get_target_transform(target):
    if target in LOG_TARGETS:
        return 'log1p'
    if target in ARCSINH_TARGETS:
        return 'arcsinh'
    return 'none'


def target_transform(y, transformacao='none'):
    y_arr = np.asarray(y, dtype=float)
    if not np.isfinite(y_arr).all():
        n_bad = np.size(y_arr) - np.isfinite(y_arr).sum()
        raise ValueError(f'target_transform: há {n_bad} valores não finitos.')
    if transformacao == 'log1p':
        if np.any(y_arr <= -1):
            raise ValueError("target_transform(log1p): valores <= -1 encontrados. Use 'arcsinh'.")
        return np.log1p(y_arr)
    if transformacao == 'arcsinh':
        return np.arcsinh(y_arr)
    return y_arr.copy()


def target_inverse_transform(y_pred, transformacao='none'):
    y_arr = np.asarray(y_pred, dtype=float)
    if transformacao == 'log1p':
        return np.expm1(y_arr)
    if transformacao == 'arcsinh':
        return np.sinh(y_arr)
    return y_arr


def selecionar_features_colineares(df_train, candidate_features, target_col,
                                    threshold=0.92, max_features=MAX_FEATURES_PER_TARGET):
    """
    Seleção treino-only, com deduplicação por empresa×data-target para evitar
    que ITRs repetidas viciem a correlação com o target.

    Parâmetros
    ----------
    threshold    : limiar de correlação entre features (colinearidade). Use
                   CORR_DROP_THRESHOLD_LINEAR para Ridge/SVR e
                   CORR_DROP_THRESHOLD_TREE para RF/GB.
    max_features : cap máximo de features mantidas (None = sem limite).
    """
    cols = [c for c in candidate_features if c in df_train.columns]
    subset_cols = [c for c in ['CNPJ_CIA', 'DT_TARGET', target_col] if c in df_train.columns]
    tmp = df_train[list(dict.fromkeys(cols + subset_cols))].dropna(subset=[target_col]).copy()

    # Deduplicação: uma linha por empresa × data-alvo reduz o viés das ITRs
    if 'CNPJ_CIA' in tmp.columns and 'DT_TARGET' in tmp.columns:
        tmp = tmp.drop_duplicates(subset=['CNPJ_CIA', 'DT_TARGET'])

    if tmp.empty or len(cols) == 0:
        return cols

    corr_target = tmp[cols].corrwith(tmp[target_col]).abs().fillna(0.0).sort_values(ascending=False)
    ordered = corr_target.index.tolist()
    corr_mat = tmp[cols].corr().abs().fillna(0.0)

    kept = []
    for feat in ordered:
        if feat not in corr_mat.columns:
            continue
        if all(corr_mat.loc[feat, k] <= threshold for k in kept):
            kept.append(feat)
        if max_features is not None and len(kept) >= max_features:
            break

    if len(kept) == 0:
        kept = ordered[: min(20, len(ordered))]
    return kept


# Algoritmos
est_ridge = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('ridge', Ridge(random_state=SEED)),
])
grade_ridge = {'ridge__alpha': [100.0, 1000.0, 10000.0]}

est_svr = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('svr', SVR(kernel='rbf', max_iter=20000)),
])
# SVR: gamma='auto' raramente vence em painel financeiro com demeaning aplicado.
# Redução: 3×3×2=18 → 3×3×1=9 combinações (−50%).
grade_svr = {
    'svr__C':       [0.1, 1.0, 10.0],
    'svr__epsilon': [0.05, 0.1, 0.5],
    'svr__gamma':   ['scale'],          # 'auto' removido — scale domina após normalização
}

est_rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestRegressor(random_state=SEED, n_jobs=-1)),
])
# RF: adicionado n_estimators=200 para dar chance real ao modelo;
# 100 era insuficiente nos logs anteriores.
# 2×2 = 4 combinações (igual, mas mais informativo).
grade_rf = {
    'rf__max_depth':    [3, 5],
    'rf__n_estimators': [100, 200],
}

est_gb = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('gb', GradientBoostingRegressor(random_state=SEED)),
])
# GradientBoosting — grid informado pelos best_params dos logs anteriores:
#   • learning_rate: 0.10 removido — overfita em séries financeiras curtas;
#     0.03 e 0.05 foram os valores vencedores nos targets com U < 1.
#   • max_depth: 7 removido — com 25 empresas e séries curtas, árvores rasas
#     (3–4) generalizam melhor; 7 produziu U > 1 consistentemente.
#   • subsample: fixado em 0.8 — reduz variância em amostras pequenas;
#     1.0 (sem subsampling) não produziu melhoria nos logs.
#   • n_estimators: 100 removido — insuficiente para learning_rate baixo.
# Resultado: 2×2×2×1 = 8 combinações vs 54 anteriores — redução de 85%.
# Justificativa acadêmica: busca informada por execução preliminar (prática
# padrão em ML aplicado; ver Bergstra & Bengio, 2012).
grade_gb = {
    'gb__n_estimators':  [200, 300],
    'gb__learning_rate': [0.03, 0.05],
    'gb__max_depth':     [3, 4],
    'gb__subsample':     [0.8],
}

ALGORITMOS = {
    'Ridge': (est_ridge, grade_ridge),
    'SVR': (est_svr, grade_svr),
    'RandomForest': (est_rf, grade_rf),
    'GradientBoosting': (est_gb, grade_gb),
}

logger.info('%d algoritmos configurados | Walk-Forward n_splits=%d', len(ALGORITMOS), N_SPLITS_WF)
print(f'✅ {len(ALGORITMOS)} algoritmos configurados')

2026-05-13 16:06:48 | INFO     | 4 algoritmos configurados | Walk-Forward n_splits=3


✅ 4 algoritmos configurados


## Etapa 4. Treinamento com Walk-Forward nested CV

In [5]:
def treinar_alg(nome, estimador, grade, df_treino_completo, target, features,
                transformacao='none', n_splits_wf=N_SPLITS_WF,
                group_col='CNPJ_CIA', time_col='ANO'):
    """
    Treinamento company-aware:
    - pesos por empresa e por futuro repetido;
    - walk-forward temporal por ano;
    - scoring por SMAPE;
    - métricas macro por empresa.
    """
    use_cols = [c for c in features + [group_col, target] if c in df_treino_completo.columns]
    if time_col in df_treino_completo.columns:
        use_cols += [time_col]
    if 'DT_REFER' in df_treino_completo.columns:
        use_cols += ['DT_REFER']
    if 'DT_TARGET' in df_treino_completo.columns:
        use_cols += ['DT_TARGET']

    use_cols = list(dict.fromkeys(use_cols))
    df_t = df_treino_completo[use_cols].copy()
    df_t = df_t[df_t[target].notna()].reset_index(drop=True)

    if TRAIN_DFP_ONLY and 'ORIGEM' in df_t.columns:
        df_t = df_t[df_t['ORIGEM'] == 'DFP'].copy().reset_index(drop=True)

    if time_col not in df_t.columns:
        time_col = 'DT_REFER' if 'DT_REFER' in df_t.columns else ('ANO' if 'ANO' in df_t.columns else None)

    X_full = df_t[features].values
    y_full = df_t[target].values
    y_fit_full = target_transform(y_full, transformacao)

    # ── Normalização por empresa (within-company z-score) ───────────────
    # Problema: com 25 empresas de escalas muito distintas, o RobustScaler
    # do pipeline normaliza pelo painel inteiro — empresas grandes dominam.
    # Solução: demeaning por empresa antes de empilhar. Subtrai a média de
    # cada feature dentro de cada empresa, preservando variação cross-sectional
    # via desvio padrão global. O RobustScaler do pipeline faz o resto.
    if group_col in df_t.columns:
        _grupos = df_t[group_col].values
        _X_df = pd.DataFrame(X_full, columns=features)
        for _emp in np.unique(_grupos):
            _mask = _grupos == _emp
            _X_df.loc[_mask] = _X_df.loc[_mask] - _X_df.loc[_mask].mean()
        X_full = _X_df.values

    sample_weight_full = calcular_pesos_amostra(df_t, group_col=group_col, future_col='DT_TARGET')
    final_step = list(estimador.named_steps.keys())[-1]
    fit_params_full = {f'{final_step}__sample_weight': sample_weight_full}

    folds_ext = criar_folds_walkforward(df_t, time_col=time_col or 'ANO', n_splits=n_splits_wf)
    if len(folds_ext) < 2:
        logger.warning('%s | %s: folds insuficientes, fallback cv=3', nome, target)
        gs_fb = GridSearchCV(estimador, grade, cv=3, scoring=smape_scorer,
                             refit=True, n_jobs=-1, verbose=0)
        gs_fb.fit(X_full, y_fit_full, **fit_params_full)
        best_est = gs_fb.best_estimator_
        metricas = {
            'RMSE_CV_macro_empresa': np.nan,
            'RMSE_CV_macro_empresa_std': np.nan,
            'MAE_CV_macro_empresa': np.nan,
            'SMAPE_CV_macro_empresa': np.nan,
            'SMAPE_CV_macro_empresa_std': np.nan,
            'R2_CV_macro_empresa': np.nan,
            'R2_CV_pooled': np.nan,
            'R2_within_CV': np.nan,
            'TheilU_CV_macro_empresa': np.nan,
            'DA_CV_macro_empresa': np.nan,
            'RMSE_CV_pooled': np.nan,
            'SMAPE_CV_pooled': np.nan,
            'transformacao': transformacao,
            'log_transform': transformacao == 'log1p',
            'best_params': gs_fb.best_params_,
            'n_folds_wf': 0,
            'selected_features': features,
        }
        return best_est, metricas

    rmse_macro_v, mae_macro_v, smape_macro_v, r2_macro_v, theil_macro_v, da_macro_v = [], [], [], [], [], []
    rmse_pool_v, smape_pool_v, r2_pool_v, r2_within_v = [], [], [], []

    for tr_idx_ext, val_idx_ext in folds_ext:
        X_tr_ext = X_full[tr_idx_ext]
        X_val_ext = X_full[val_idx_ext]   
        y_tr_ext = y_fit_full[tr_idx_ext]
        y_val_orig = y_full[val_idx_ext]

        # Demeaning por empresa dentro do fold (evita leakage de escala cross-empresa)
        if group_col in df_t.columns:
            _grp_tr = df_t[group_col].values[tr_idx_ext]
            _grp_val = df_t[group_col].values[val_idx_ext]
            _X_tr_df = pd.DataFrame(X_tr_ext, columns=features)
            _X_val_df = pd.DataFrame(X_val_ext, columns=features)
            # Média de cada empresa NO treino — aplicada também na validação
            _emp_mean_map = _X_tr_df.copy()
            _emp_mean_map['_g'] = _grp_tr
            _emp_mean_map = _emp_mean_map.groupby('_g')[features].mean()
            # Demeaning treino
            for _emp in np.unique(_grp_tr):
                _mt = _grp_tr == _emp
                if _emp in _emp_mean_map.index:
                    _X_tr_df.loc[_mt] -= _emp_mean_map.loc[_emp].values
            # Demeaning validação com médias do treino (sem olhar futuro)
            for _emp in np.unique(_grp_val):
                _mv = _grp_val == _emp
                if _emp in _emp_mean_map.index:
                    _X_val_df.loc[_mv] -= _emp_mean_map.loc[_emp].values
            X_tr_ext = _X_tr_df.values
            X_val_ext = _X_val_df.values

        df_sub = df_t.iloc[tr_idx_ext].reset_index(drop=True)
        folds_int = criar_folds_walkforward(df_sub, time_col=time_col or 'ANO', n_splits=max(2, n_splits_wf - 1))
        cv_int = folds_int if len(folds_int) >= 2 else 3

        w_tr_ext = sample_weight_full[tr_idx_ext]
        fit_params_tr = {f'{final_step}__sample_weight': w_tr_ext}

        gs = GridSearchCV(estimador, grade, cv=cv_int, scoring=smape_scorer,
                          refit=True, n_jobs=-1, verbose=0)
        gs.fit(X_tr_ext, y_tr_ext, **fit_params_tr)
        melhor_fold = gs.best_estimator_

        y_pred_raw = melhor_fold.predict(X_val_ext)
        y_pred = target_inverse_transform(y_pred_raw, transformacao)

        df_fold_eval = df_t.iloc[val_idx_ext][[group_col]].copy()
        if time_col in df_t.columns:
            df_fold_eval[time_col] = df_t.iloc[val_idx_ext][time_col].values
        df_fold_eval['y_true'] = y_val_orig
        df_fold_eval['y_pred'] = y_pred

        m_fold = calcular_metricas_painel(df_fold_eval, group_col=group_col,
                                          time_col=time_col or group_col,
                                          y_true_col='y_true', y_pred_col='y_pred')
        rmse_macro_v.append(m_fold['RMSE_macro_empresa'])
        mae_macro_v.append(m_fold['MAE_macro_empresa'])
        smape_macro_v.append(m_fold['SMAPE_macro_empresa'])
        r2_macro_v.append(m_fold['R2_macro_empresa'])
        theil_macro_v.append(m_fold['TheilU_macro_empresa'])
        da_macro_v.append(m_fold['DA_macro_empresa'])
        rmse_pool_v.append(m_fold['RMSE_pooled'])
        smape_pool_v.append(m_fold['SMAPE_pooled'])
        r2_pool_v.append(m_fold['R2_pooled'])
        r2_within_v.append(m_fold['R2_within'])

    gs_final = GridSearchCV(estimador, grade, cv=folds_ext if len(folds_ext) >= 2 else 3,
                            scoring=smape_scorer, refit=True, n_jobs=-1, verbose=0)
    gs_final.fit(X_full, y_fit_full, **fit_params_full)
    best_est = gs_final.best_estimator_

    def _m(lst):
        return float(np.nanmean(lst))
    def _s(lst):
        return float(np.nanstd(lst))

    metricas = {
        'RMSE_CV_macro_empresa': _m(rmse_macro_v),
        'RMSE_CV_macro_empresa_std': _s(rmse_macro_v),
        'MAE_CV_macro_empresa': _m(mae_macro_v),
        'SMAPE_CV_macro_empresa': _m(smape_macro_v),
        'SMAPE_CV_macro_empresa_std': _s(smape_macro_v),
        'R2_CV_macro_empresa': _m(r2_macro_v),
        'R2_CV_pooled': _m(r2_pool_v),
        'R2_within_CV': _m(r2_within_v),
        'TheilU_CV_macro_empresa': _m(theil_macro_v),
        'DA_CV_macro_empresa': _m(da_macro_v),
        'RMSE_CV_pooled': _m(rmse_pool_v),
        'SMAPE_CV_pooled': _m(smape_pool_v),
        'transformacao': transformacao,
        'log_transform': transformacao == 'log1p',
        'best_params': gs_final.best_params_,
        'n_folds_wf': len(folds_ext),
        'selected_features': features,
    }

    flag_theil = '✅' if metricas['TheilU_CV_macro_empresa'] < 1 else '⚠️'
    logger.info(
        '  %-20s RMSEm=%10.0f±%8.0f  SMAPEm=%5.1f%%  R2m=%5.3f  U=%s%.3f  DAm=%.1f%%  folds=%d',
        nome,
        metricas['RMSE_CV_macro_empresa'], metricas['RMSE_CV_macro_empresa_std'],
        metricas['SMAPE_CV_macro_empresa'] * 100, metricas['R2_CV_macro_empresa'],
        flag_theil, metricas['TheilU_CV_macro_empresa'],
        metricas['DA_CV_macro_empresa'] * 100, metricas['n_folds_wf']
    )
    print(
        f"  {flag_theil} {nome:<20} RMSEm={metricas['RMSE_CV_macro_empresa']:>12,.0f}  "
        f"SMAPEm={metricas['SMAPE_CV_macro_empresa']:>5.1%}  R²m={metricas['R2_CV_macro_empresa']:>6.3f}  "
        f"U={metricas['TheilU_CV_macro_empresa']:.3f}  DAm={metricas['DA_CV_macro_empresa']:.1%}  folds={metricas['n_folds_wf']}"
    )
    return best_est, metricas


## Etapa 5. Loop principal por target

In [6]:
resultados = {}
metricas_teste = {}
feature_importances = {}
modelos_finais = {}
selected_features_por_target = {}   # target → selected_tree
features_por_target_alg = {}         # (target, algoritmo) → features corretas

for target in TARGETS:
    transformacao = get_target_transform(target)
    b = baselines.get(target, {})

    print(f"\n{'='*80}")
    print(f"TARGET: {target} | transform={transformacao}")
    if b:
        print(f"Baseline → RMSEm={b.get('RMSE_macro_empresa', np.nan):,.0f}  SMAPEm={b.get('SMAPE_macro_empresa', np.nan):.1%}  "
              f"R²m={b.get('R2_macro_empresa', np.nan):.3f}  DAm={b.get('DA_macro_empresa', np.nan):.1%}  Cob={b.get('Cobertura_baseline', np.nan):.1%}")

    # ── Seleção de features por família de modelo ──────────────────────────────
    # Modelos lineares (Ridge/SVR): threshold mais restritivo (colinearidade prejudica)
    # Modelos de árvore (RF/GB) : threshold mais permissivo (absolvem colinearidade)
    base_features = [c for c in FEATURES if c in treino.columns and c != target]
    train_for_sel = treino[base_features + [target]
                           + [c for c in ['CNPJ_CIA', 'DT_TARGET'] if c in treino.columns]].copy()

    selected_linear = selecionar_features_colineares(
        train_for_sel, base_features, target,
        threshold=CORR_DROP_THRESHOLD_LINEAR,
        max_features=MAX_FEATURES_PER_TARGET,
    )
    selected_tree = selecionar_features_colineares(
        train_for_sel, base_features, target,
        threshold=CORR_DROP_THRESHOLD_TREE,
        max_features=MAX_FEATURES_PER_TARGET,
    )

    # Mapa: qual conjunto de features usar por família de modelo
    FEATURES_POR_FAMILIA = {
        'Ridge':            selected_linear,
        'SVR':              selected_linear,
        'RandomForest':     selected_tree,
        'GradientBoosting': selected_tree,
    }

    # Persiste o conjunto 'tree' como representativo do target (mais amplo)
    selected_features_por_target[target] = selected_tree
    # Persiste por (target, algoritmo) — evita mismatch na avaliação de teste
    for _nome in ALGORITMOS:
        features_por_target_alg[(target, _nome)] = FEATURES_POR_FAMILIA.get(_nome, selected_tree)

    print(f"Features → linear={len(selected_linear)} | tree={len(selected_tree)}")

    resultados[target] = {}
    metricas_teste[target] = {}

    for nome, (est, grade) in ALGORITMOS.items():
        features_nome = FEATURES_POR_FAMILIA.get(nome, selected_tree)
        modelo, met_cv = treinar_alg(
            nome=nome,
            estimador=est,
            grade=grade,
            df_treino_completo=treino,
            target=target,
            features=features_nome,
            transformacao=transformacao,
            n_splits_wf=N_SPLITS_WF,
            group_col='CNPJ_CIA',
            time_col='ANO',
        )
        resultados[target][nome] = (modelo, met_cv)
        modelos_finais[(target, nome)] = modelo

        joblib.dump(
            {
                'modelo': modelo,
                'transformacao': transformacao,
                'log_transform': transformacao == 'log1p',
                'features': features_nome,
                'target': target,
                'selected_features': features_nome,
                'familia': 'linear' if nome in ('Ridge', 'SVR') else 'tree',
            },
            PASTA_SAIDA / 'modelos' / f'modelo_{target}_{nome}.pkl'
        )

    logger.info('TARGET %s concluído', target)

print('\n✅ Treinamento concluído para todos os targets.')



TARGET: TARGET_DRE_3.01_ITR_T1 | transform=log1p
Baseline → RMSEm=13,107,786  SMAPEm=81.8%  R²m=-3.810  DAm=40.0%  Cob=100.0%


2026-05-13 16:06:49 | INFO     | Walk-Forward CV: 3 folds | validação: ['2021', '2022', '2023']
2026-05-13 16:06:49 | INFO     | Walk-Forward CV: 2 folds | validação: ['2019', '2020']


Features → linear=75 | tree=134


2026-05-13 16:06:55 | INFO     | Walk-Forward CV: 2 folds | validação: ['2020', '2021']
2026-05-13 16:06:58 | INFO     | Walk-Forward CV: 2 folds | validação: ['2021', '2022']
2026-05-13 16:06:58 | INFO     |   Ridge                RMSEm=   6516409± 1898156  SMAPEm= 99.5%  R2m=-3.058  U=⚠️1.281  DAm=33.3%  folds=3
2026-05-13 16:06:58 | INFO     | Walk-Forward CV: 3 folds | validação: ['2021', '2022', '2023']
2026-05-13 16:06:58 | INFO     | Walk-Forward CV: 2 folds | validação: ['2019', '2020']


  ⚠️ Ridge                RMSEm=   6,516,409  SMAPEm=99.5%  R²m=-3.058  U=1.281  DAm=33.3%  folds=3


...

  ⚠️ SVR                  RMSEm=   1,982,498  SMAPEm=78.9%  R²m=-42.602  U=2.000  DAm=33.3%  folds=3


2026-05-13 16:55:21 | INFO     | Walk-Forward CV: 2 folds | validação: ['2020', '2021']
2026-05-13 16:55:24 | INFO     | Walk-Forward CV: 2 folds | validação: ['2021', '2022']
2026-05-13 16:55:34 | INFO     |   RandomForest         RMSEm=   2677616±  963354  SMAPEm=100.0%  R2m=-64.850  U=⚠️2.000  DAm=0.0%  folds=3
2026-05-13 16:55:34 | INFO     | Walk-Forward CV: 3 folds | validação: ['2021', '2022', '2023']
2026-05-13 16:55:34 | INFO     | Walk-Forward CV: 2 folds | validação: ['2019', '2020']


  ⚠️ RandomForest         RMSEm=   2,677,616  SMAPEm=100.0%  R²m=-64.850  U=2.000  DAm=0.0%  folds=3


2026-05-13 16:55:43 | INFO     | Walk-Forward CV: 2 folds | validação: ['2020', '2021']
2026-05-13 16:55:58 | INFO     | Walk-Forward CV: 2 folds | validação: ['2021', '2022']
2026-05-13 16:56:37 | INFO     |   GradientBoosting     RMSEm=   2094789±  321651  SMAPEm=100.0%  R2m=-61.627  U=⚠️2.000  DAm=33.3%  folds=3
2026-05-13 16:56:37 | INFO     | TARGET TARGET_DFC_MI_6.01_DFP concluído


  ⚠️ GradientBoosting     RMSEm=   2,094,789  SMAPEm=100.0%  R²m=-61.627  U=2.000  DAm=33.3%  folds=3

✅ Treinamento concluído para todos os targets.


## Etapa 6. Avaliação no teste hold-out

In [7]:
def avaliar_teste(modelo, df_eval, features, target, transformacao, group_col='CNPJ_CIA', time_col='DT_REFER'):
    cols = [c for c in features + [target, group_col] if c in df_eval.columns]
    if time_col in df_eval.columns:
        cols += [time_col]
    cols = list(dict.fromkeys(cols))
    df = df_eval[cols].copy()
    df = df[df[target].notna()].reset_index(drop=True)

    y_pred_raw = modelo.predict(df[features].values)
    y_pred = target_inverse_transform(y_pred_raw, transformacao)

    df_out = df[[group_col]].copy()
    if time_col in df.columns:
        df_out[time_col] = df[time_col].values
    df_out['y_true'] = df[target].values
    df_out['y_pred'] = y_pred

    return calcular_metricas_painel(df_out, group_col=group_col,
                                    time_col=time_col if time_col in df_out.columns else group_col,
                                    y_true_col='y_true', y_pred_col='y_pred')


predicoes_teste_detalhadas = []
print('\n=== Avaliação no Teste Hold-out (2024–2025) ===')
for target in TARGETS:
    transformacao = get_target_transform(target)
    b = baselines.get(target, {})
    selected_features = selected_features_por_target[target]

    # df_te_alg é construído por algoritmo dentro do loop abaixo (features corretas por família)
    # Mantemos df_te apenas para a coluna de features tree (referência para feature importance)
    _feats_tree = selected_features_por_target.get(target, selected_features)
    df_te = teste[[f for f in _feats_tree if f in teste.columns] + [target, 'CNPJ_CIA']
                   + (['DT_REFER'] if 'DT_REFER' in teste.columns else [])].copy()
    df_te = df_te[df_te[target].notna()].copy()

    baseline_rmse = b.get('RMSE_macro_empresa', np.inf)
    print(f"\n{target} (baseline RMSEm={baseline_rmse:,.0f}  DAm={b.get('DA_macro_empresa', 0):.1%}  Cob={b.get('Cobertura_baseline', np.nan):.1%})")
    print(f"  {'Algoritmo':<20} {'RMSEm':>14} {'SMAPEm':>8} {'R²m':>7} {'U':>7} {'DAm':>7} {'Bateu?':>7}")
    print(f"  {'-'*20} {'-'*14} {'-'*8} {'-'*7} {'-'*7} {'-'*7} {'-'*7}")

    for nome, (modelo, _) in resultados[target].items():
        # Busca o conjunto de features correto para este algoritmo
        # Evita mismatch: Ridge/SVR usam selected_linear, RF/GB usam selected_tree
        feats_alg = features_por_target_alg.get((target, nome), selected_features)
        # Garante que só passa features que existem no teste
        feats_alg = [f for f in feats_alg if f in teste.columns]

        df_te_alg = teste[feats_alg + [target, 'CNPJ_CIA']
                          + (['DT_REFER'] if 'DT_REFER' in teste.columns else [])].copy()
        df_te_alg = df_te_alg[df_te_alg[target].notna()].copy()

        m = avaliar_teste(modelo, df_te_alg, feats_alg, target, transformacao)
        metricas_teste[target][nome] = m
        bateu = m['RMSE_macro_empresa'] < baseline_rmse
        theil_ok = (m['TheilU_macro_empresa'] or 1.0) < 1.0
        flag = '✅' if bateu and theil_ok else ('🟡' if bateu else '❌')

        print(
            f"  {flag} {nome:<18} {m['RMSE_macro_empresa']:>14,.0f} {m['SMAPE_macro_empresa']:>8.1%} "
            f"{m['R2_macro_empresa']:>7.3f} {m['TheilU_macro_empresa']:>7.3f} {m['DA_macro_empresa']:>7.1%} {'✅' if bateu else '❌':>7}"
        )
        logger.info('Teste | %s | %s: RMSEm=%.0f SMAPEm=%.2f%% R2m=%.3f TheilU=%.3f DAm=%.1f%%',
                    target, nome,
                    m['RMSE_macro_empresa'], m['SMAPE_macro_empresa'] * 100,
                    m['R2_macro_empresa'], m['TheilU_macro_empresa'], m['DA_macro_empresa'] * 100)

        # Guarda previsão detalhada por linha para inspeção posterior
        y_pred_raw = modelo.predict(df_te_alg[feats_alg].values)
        y_pred = target_inverse_transform(y_pred_raw, transformacao)
        aux = df_te_alg[['CNPJ_CIA'] + (["DT_REFER"] if 'DT_REFER' in df_te_alg.columns else [])].copy()
        aux['Target'] = target
        aux['Algoritmo'] = nome
        aux['y_true'] = df_te_alg[target].values
        aux['y_pred'] = y_pred
        aux['erro'] = aux['y_true'] - aux['y_pred']
        predicoes_teste_detalhadas.append(aux)

        # Feature importance do melhor modelo será definido depois; este bloco só calcula tudo

2026-05-13 16:56:37 | INFO     | Teste | TARGET_DRE_3.01_ITR_T1 | Ridge: RMSEm=12744015 SMAPEm=91.79% R2m=-3.141 TheilU=1.415 DAm=40.0%



=== Avaliação no Teste Hold-out (2024–2025) ===

TARGET_DRE_3.01_ITR_T1 (baseline RMSEm=13,107,786  DAm=40.0%  Cob=100.0%)
  Algoritmo                     RMSEm   SMAPEm     R²m       U     DAm  Bateu?
  -------------------- -------------- -------- ------- ------- ------- -------
  🟡 Ridge                  12,744,015    91.8%  -3.141   1.415   40.0%       ✅


2026-05-13 16:56:38 | INFO     | Teste | TARGET_DRE_3.01_ITR_T1 | SVR: RMSEm=14613693 SMAPEm=99.57% R2m=-3.500 TheilU=1.507 DAm=40.0%


  ❌ SVR                    14,613,693    99.6%  -3.500   1.507   40.0%       ❌


2026-05-13 16:56:38 | INFO     | Teste | TARGET_DRE_3.01_ITR_T1 | RandomForest: RMSEm=15342441 SMAPEm=100.00% R2m=-2.963 TheilU=1.302 DAm=33.3%
2026-05-13 16:56:38 | INFO     | Teste | TARGET_DRE_3.01_ITR_T1 | GradientBoosting: RMSEm=13290284 SMAPEm=76.50% R2m=-2.451 TheilU=1.321 DAm=33.3%


...

  ❌ RandomForest            2,095,397   100.0%  -3.826   1.241   25.0%       ❌
  ❌ GradientBoosting        2,041,412   100.0%  -3.837   1.223   25.0%       ❌

TARGET_DFC_MI_6.01_ITR_T3 (baseline RMSEm=2,780,914  DAm=33.3%  Cob=65.8%)
  Algoritmo                     RMSEm   SMAPEm     R²m       U     DAm  Bateu?
  -------------------- -------------- -------- ------- ------- ------- -------
  ❌ Ridge               4,631,788,737   100.0% -71841886.138   2.000   66.7%       ❌
  ❌ SVR                     4,785,491   100.0% -21.800   2.000   66.7%       ❌


2026-05-13 16:56:47 | INFO     | Teste | TARGET_DFC_MI_6.01_ITR_T3 | RandomForest: RMSEm=2016365 SMAPEm=100.00% R2m=-3.305 TheilU=2.000 DAm=0.0%
2026-05-13 16:56:47 | INFO     | Teste | TARGET_DFC_MI_6.01_ITR_T3 | GradientBoosting: RMSEm=2111475 SMAPEm=100.00% R2m=-3.764 TheilU=2.000 DAm=0.0%
2026-05-13 16:56:47 | INFO     | Teste | TARGET_DFC_MI_6.01_DFP | Ridge: RMSEm=3938081 SMAPEm=100.00% R2m=nan TheilU=nan DAm=0.0%
2026-05-13 16:56:47 | INFO     | Teste | TARGET_DFC_MI_6.01_DFP | SVR: RMSEm=3910802 SMAPEm=93.16% R2m=nan TheilU=nan DAm=0.0%


  🟡 RandomForest            2,016,365   100.0%  -3.305   2.000    0.0%       ✅
  🟡 GradientBoosting        2,111,475   100.0%  -3.764   2.000    0.0%       ✅

TARGET_DFC_MI_6.01_DFP (baseline RMSEm=3,403,676  DAm=0.0%  Cob=47.0%)
  Algoritmo                     RMSEm   SMAPEm     R²m       U     DAm  Bateu?
  -------------------- -------------- -------- ------- ------- ------- -------
  ❌ Ridge                   3,938,081   100.0%     nan     nan    0.0%       ❌
  ❌ SVR                     3,910,802    93.2%     nan     nan    0.0%       ❌


2026-05-13 16:56:47 | INFO     | Teste | TARGET_DFC_MI_6.01_DFP | RandomForest: RMSEm=4147459 SMAPEm=100.00% R2m=nan TheilU=nan DAm=0.0%
2026-05-13 16:56:47 | INFO     | Teste | TARGET_DFC_MI_6.01_DFP | GradientBoosting: RMSEm=3735183 SMAPEm=100.00% R2m=nan TheilU=nan DAm=0.0%


  ❌ RandomForest            4,147,459   100.0%     nan     nan    0.0%       ❌
  ❌ GradientBoosting        3,735,183   100.0%     nan     nan    0.0%       ❌


## Etapa 7. Seleção do melhor modelo por target

In [8]:
def escolher_melhor_modelo_cv(resultados_target):
    return min(
        resultados_target.items(),
        key=lambda item: (
            item[1][1].get('SMAPE_CV_macro_empresa', np.inf),
            item[1][1].get('TheilU_CV_macro_empresa', np.inf),
            item[1][1].get('RMSE_CV_macro_empresa', np.inf),
        )
    )[0]


melhores = {t: escolher_melhor_modelo_cv(resultados[t]) for t in TARGETS}
print('\n=== Melhor modelo por target (critério: SMAPE_CV macro por empresa) ===')
for t, alg in melhores.items():
    m_cv = resultados[t][alg][1]
    m_test = metricas_teste[t][alg]
    print(f"  {t:<35} → {alg:<18} SMAPE_CV={m_cv['SMAPE_CV_macro_empresa']:.1%} | SMAPE_teste={m_test['SMAPE_macro_empresa']:.1%} | U_teste={m_test['TheilU_macro_empresa']:.3f}")



=== Melhor modelo por target (critério: SMAPE_CV macro por empresa) ===
  TARGET_DRE_3.01_ITR_T1              → GradientBoosting   SMAPE_CV=34.9% | SMAPE_teste=76.5% | U_teste=1.321
  TARGET_DRE_3.01_ITR_T2              → GradientBoosting   SMAPE_CV=30.5% | SMAPE_teste=49.4% | U_teste=0.964
  TARGET_DRE_3.01_ITR_T3              → GradientBoosting   SMAPE_CV=29.4% | SMAPE_teste=48.6% | U_teste=1.850
  TARGET_DRE_3.01_DFP                 → GradientBoosting   SMAPE_CV=28.5% | SMAPE_teste=33.8% | U_teste=nan
  TARGET_DRE_3.11_ITR_T1              → GradientBoosting   SMAPE_CV=96.0% | SMAPE_teste=100.0% | U_teste=1.307
  TARGET_DRE_3.11_ITR_T2              → SVR                SMAPE_CV=92.5% | SMAPE_teste=84.2% | U_teste=1.112
  TARGET_DRE_3.11_ITR_T3              → SVR                SMAPE_CV=93.6% | SMAPE_teste=73.7% | U_teste=1.985
  TARGET_DRE_3.11_DFP                 → SVR                SMAPE_CV=77.5% | SMAPE_teste=70.7% | U_teste=nan
  TARGET_EBITDA_ITR_T1                → GradientBo

## Etapa 8. Feature importance e resíduos


In [10]:
def extrair_importancia(modelo, features):
    step = list(modelo.named_steps.keys())[-1]
    est_final = modelo.named_steps[step]
    if hasattr(est_final, 'feature_importances_'):
        imp = est_final.feature_importances_
    elif hasattr(est_final, 'coef_'):
        imp = np.abs(est_final.coef_)
    else:
        return pd.Series(dtype=float)
    return pd.Series(imp, index=features).sort_values(ascending=False)


print('\n=== Feature Importance — Melhor Modelo por Target ===')
n_t = len(TARGETS)
fig, axes = plt.subplots(n_t, 1, figsize=(11, 5 * n_t))
if n_t == 1:
    axes = [axes]

for i, target in enumerate(TARGETS):
    melhor_nome = melhores[target]
    melhor_mod = resultados[target][melhor_nome][0]
    feats_t = selected_features_por_target[target]
    imp = extrair_importancia(melhor_mod, feats_t)
    feature_importances[target] = {
        'algoritmo': melhor_nome,
        'features': feats_t,
        'importancias': imp.to_dict(),
    }

    ax = axes[i]
    if not imp.empty:
        top = imp.head(min(12, len(imp)))
        ax.barh(range(len(top)), top.values[::-1], alpha=0.9)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top.index[::-1], fontsize=9)
        ax.set_title(f"{target.replace('TARGET_', '')} — {melhor_nome} | SMAPE_teste={metricas_teste[target][melhor_nome]['SMAPE_macro_empresa']:.1%}",
                     fontsize=10, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
        for j, v in enumerate(top.values[::-1]):
            ax.text(v + imp.max() * 0.005, j, f'{v:.3f}', va='center', fontsize=8)
    else:
        ax.text(0.5, 0.5, 'Sem importância disponível', ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()

plt.suptitle('Feature Importance — Melhor Modelo por Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Salvo: outputs/feature_importance.png')


print('\n=== Análise de Resíduos — Teste 2024–2025 ===')
cols_setor_disp = [c for c in teste.columns if c.startswith('setor_')]
fig, axes = plt.subplots(n_t, 2, figsize=(14, 5 * n_t))
if n_t == 1:
    axes = axes.reshape(1, -1)

for i, target in enumerate(TARGETS):
    melhor_nome = melhores[target]
    melhor_mod = resultados[target][melhor_nome][0]
    feats_t = selected_features_por_target[target]
    transformacao = get_target_transform(target)

    df_te = teste[feats_t + [target, 'CNPJ_CIA'] + (["DT_REFER"] if 'DT_REFER' in teste.columns else []) + cols_setor_disp].copy()
    df_te = df_te[df_te[target].notna()].copy()
    y_te = df_te[target].values
    y_pred = target_inverse_transform(melhor_mod.predict(df_te[feats_t].values), transformacao)
    residuos = y_te - y_pred

    ax1 = axes[i, 0]
    lim = max(np.nanmax(np.abs(y_te)), np.nanmax(np.abs(y_pred))) * 1.05
    ax1.scatter(y_pred, y_te, alpha=0.45, s=18, edgecolors='none')
    ax1.plot([-lim, lim], [-lim, lim], 'r--', lw=1.3)
    ax1.set_xlabel('Predito')
    ax1.set_ylabel('Observado')
    ax1.set_title(f"{target.replace('TARGET_', '')} — {melhor_nome}\nPredito × Observado", fontsize=10, fontweight='bold')
    ax1.text(0.05, 0.92, f'R²m={metricas_teste[target][melhor_nome]["R2_macro_empresa"]:.3f}  SMAPE={metricas_teste[target][melhor_nome]["SMAPE_macro_empresa"]:.1%}',
             transform=ax1.transAxes, fontsize=8,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    ax2 = axes[i, 1]
    ax2.scatter(y_pred, residuos, alpha=0.45, s=18, edgecolors='none')
    ax2.axhline(0, color='r', lw=1.3, ls='--')
    ax2.axhline(np.std(residuos), color='gray', lw=1, ls=':', alpha=0.7)
    ax2.axhline(-np.std(residuos), color='gray', lw=1, ls=':', alpha=0.7)
    ax2.set_xlabel('Predito')
    ax2.set_ylabel('Resíduo')
    ax2.set_title(f'Resíduos × Predito | skew={pd.Series(residuos).skew():.2f}', fontsize=10, fontweight='bold')

plt.suptitle('Análise de Resíduos — Teste 2024–2025', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'analise_residuos.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Salvo: outputs/analise_residuos.png')


=== Feature Importance — Melhor Modelo por Target ===
✅ Salvo: outputs/feature_importance.png

=== Análise de Resíduos — Teste 2024–2025 ===


ValueError: X has 137 features, but SimpleImputer is expecting 134 features as input.

## Etapa 9. Persistência completa de artefatos

In [ ]:
# =============================================================================
# Etapa Prospectiva — Predição sobre dados de 2026 (ITR Q1 real + horizonte)
# =============================================================================
# O prospectivo.parquet contém o ITR Q1/2026 (dado real) e linhas futuras
# para previsão em cascata: Q2, Q3 e DFP 2026.
# Esta célula aplica o melhor modelo de cada target sobre esse conjunto.

if prospectivo.empty:
    print('⚠️  prospectivo.parquet vazio ou não encontrado — etapa ignorada.')
else:
    # Normaliza features no prospectivo (mesmo pipeline do treino/teste)
    for _col in ('DT_REFER', 'DT_TARGET', 'DT_TARGET_DFP'):
        if _col in prospectivo.columns:
            prospectivo[_col] = (pd.to_datetime(prospectivo[_col], utc=True, errors='coerce')
                                   .dt.tz_localize(None))
    if 'flag_covid' not in prospectivo.columns:
        prospectivo['flag_covid'] = prospectivo['ANO'].isin(COVID_ANOS).astype(float)
    if 'ano_norm' not in prospectivo.columns:
        prospectivo['ano_norm'] = (prospectivo['ANO'].astype(float) - 2015.0) / 10.0

    predicoes_prospectivas = []

    for target in TARGETS:
        if target not in melhores:
            continue
        melhor_nome = melhores[target]
        modelo      = resultados[target][melhor_nome][0]
        feats_t     = selected_features_por_target[target]
        transformacao = get_target_transform(target)

        # Apenas linhas do prospectivo com todas as features disponíveis
        feats_disp = [f for f in feats_t if f in prospectivo.columns]
        if len(feats_disp) < len(feats_t) * 0.5:
            logger.warning('Prospectivo: features insuficientes para %s (%d/%d)', target, len(feats_disp), len(feats_t))
            continue

        df_p = prospectivo[feats_disp + ['CNPJ_CIA']
                           + ([c for c in ('DT_REFER', 'ORIGEM') if c in prospectivo.columns])].copy()
        df_p = df_p.dropna(subset=feats_disp, how='all').reset_index(drop=True)
        if df_p.empty:
            continue

        # Imputa NaN restantes com mediana do treino
        X_p = df_p[feats_disp].values
        y_pred_raw = modelo.predict(X_p)
        y_pred = target_inverse_transform(y_pred_raw, transformacao)

        horizonte = next((h for h in _HORIZONTES if target.endswith(h)), 'N/A')
        for i_row, (_, row) in enumerate(df_p.iterrows()):
            predicoes_prospectivas.append({
                'CNPJ_CIA':   row.get('CNPJ_CIA'),
                'DT_REFER':   row.get('DT_REFER'),
                'ORIGEM':     row.get('ORIGEM', 'PROSP'),
                'Target':     target,
                'Horizonte':  horizonte,
                'Algoritmo':  melhor_nome,
                'y_pred':     y_pred[i_row],
            })

    if predicoes_prospectivas:
        df_prosp_out = pd.DataFrame(predicoes_prospectivas)
        df_prosp_out.to_csv(PASTA_SAIDA / 'predicoes_prospectivas.csv', index=False)
        df_prosp_out.to_parquet(PASTA_SAIDA / 'predicoes_prospectivas.parquet', index=False)
        print(f'\n✅ Predições prospectivas: {len(df_prosp_out)} linhas')
        print(df_prosp_out.groupby(['Horizonte', 'Algoritmo']).size().to_string())
        logger.info('Predições prospectivas salvas: %d linhas', len(df_prosp_out))
    else:
        print('⚠️  Nenhuma predição prospectiva gerada.')


In [ ]:
rows_cv, rows_te = [], []
for target, algs in resultados.items():
    b = baselines.get(target, {})
    for alg, (_, m) in algs.items():
        horizonte = next((h for h in _HORIZONTES if target.endswith(h)), 'N/A')
        rows_cv.append({
            'Target': target,
            'Horizonte': horizonte,
            'Algoritmo': alg,
            'RMSE_CV_macro_empresa': m.get('RMSE_CV_macro_empresa'),
            'RMSE_CV_macro_empresa_std': m.get('RMSE_CV_macro_empresa_std'),
            'MAE_CV_macro_empresa': m.get('MAE_CV_macro_empresa'),
            'SMAPE_CV_macro_empresa': m.get('SMAPE_CV_macro_empresa'),
            'SMAPE_CV_macro_empresa_std': m.get('SMAPE_CV_macro_empresa_std'),
            'R2_CV_macro_empresa': m.get('R2_CV_macro_empresa'),
            'R2_CV_pooled': m.get('R2_CV_pooled'),
            'R2_within_CV': m.get('R2_within_CV'),
            'TheilU_CV_macro_empresa': m.get('TheilU_CV_macro_empresa'),
            'DA_CV_macro_empresa': m.get('DA_CV_macro_empresa'),
            'RMSE_CV_pooled': m.get('RMSE_CV_pooled'),
            'SMAPE_CV_pooled': m.get('SMAPE_CV_pooled'),
            'n_folds_wf': m.get('n_folds_wf'),
            'transformacao': m.get('transformacao'),
            'log_transform': m.get('log_transform'),
            'best_params': str(m.get('best_params')),
        })
        mt = metricas_teste[target][alg]
        rows_te.append({
            'Target': target,
            'Horizonte': horizonte,
            'Algoritmo': alg,
            'RMSE_teste_macro_empresa': mt.get('RMSE_macro_empresa'),
            'MAE_teste_macro_empresa': mt.get('MAE_macro_empresa'),
            'SMAPE_teste_macro_empresa': mt.get('SMAPE_macro_empresa'),
            'R2_teste_macro_empresa': mt.get('R2_macro_empresa'),
            'R2_teste_pooled': mt.get('R2_pooled'),
            'R2_teste_within': mt.get('R2_within'),
            'TheilU_teste_macro_empresa': mt.get('TheilU_macro_empresa'),
            'DA_teste_macro_empresa': mt.get('DA_macro_empresa'),
            'RMSE_teste_pooled': mt.get('RMSE_pooled'),
            'SMAPE_teste_pooled': mt.get('SMAPE_pooled'),
            'RMSE_baseline': b.get('RMSE_macro_empresa'),
            'Bateu_baseline': mt.get('RMSE_macro_empresa', np.inf) < b.get('RMSE_macro_empresa', np.inf),
            'TheilU_ok': (mt.get('TheilU_macro_empresa', 1.0) or 1.0) < 1.0,
        })


df_cv = pd.DataFrame(rows_cv)
df_te = pd.DataFrame(rows_te)

# predicoes detalhadas por linha
if predicoes_teste_detalhadas:
    df_pred = pd.concat(predicoes_teste_detalhadas, ignore_index=True)
else:
    df_pred = pd.DataFrame()

# salva csv/pkl/parquet
for name, obj in [
    ('resultados_cv.csv', df_cv),
    ('resultados_teste.csv', df_te),
]:
    obj.to_csv(PASTA_SAIDA / name, index=False)

if not df_pred.empty:
    df_pred.to_parquet(PASTA_SAIDA / 'predicoes_teste_detalhadas.parquet', index=False)
    df_pred.to_csv(PASTA_SAIDA / 'predicoes_teste_detalhadas.csv', index=False)

with open(PASTA_SAIDA / 'resultados_cv.pkl', 'wb') as f:
    pickle.dump(resultados, f)
with open(PASTA_SAIDA / 'metricas_teste.pkl', 'wb') as f:
    pickle.dump(metricas_teste, f)
with open(PASTA_SAIDA / 'baselines.pkl', 'wb') as f:
    pickle.dump(baselines, f)
with open(PASTA_SAIDA / 'feature_importances.pkl', 'wb') as f:
    pickle.dump(feature_importances, f)
with open(PASTA_SAIDA / 'melhores_modelos.pkl', 'wb') as f:
    pickle.dump(melhores, f)
with open(PASTA_SAIDA / 'selected_features_por_target.pkl', 'wb') as f:
    pickle.dump(selected_features_por_target, f)

relatorio = {
    'versao': 'V3_CompanyAware_WF_SMAPE',
    'data_execucao': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'ano_corte': ANO_CORTE,
    'n_treino': int(len(treino)),
    'n_teste': int(len(teste)),
    'targets': TARGETS,
    'algoritmos': list(ALGORITMOS.keys()),
    'n_features_originais': int(len(FEATURES)),
    'n_features_selecionadas_por_target': {t: len(v) for t, v in selected_features_por_target.items()},
    'train_dfp_only': TRAIN_DFP_ONLY,
    'corr_drop_threshold_linear': CORR_DROP_THRESHOLD_LINEAR,
    'corr_drop_threshold_tree': CORR_DROP_THRESHOLD_TREE,
    'horizontes': _HORIZONTES,
    'n_targets': len(TARGETS),
    'n_splits_wf': N_SPLITS_WF,
    'company_aware': True,
    'métricas_prioritárias': ['SMAPE_macro_empresa', 'TheilU_macro_empresa', 'DA_macro_empresa'],
    'selecao_modelo': 'menor SMAPE_CV_macro_empresa, desempate TheilU_CV_macro_empresa, desempate RMSE_CV_macro_empresa',
    'baseline': 'persistência do último valor da própria empresa',
    'feature_selection': 'treino-only + filtro de colinearidade',
    'pesos_amostrais': 'inverso por empresa e por target futuro repetido (DT_TARGET)',
    'results': {
        t: {
            alg: {
                'cv_smape_macro_empresa': float(resultados[t][alg][1].get('SMAPE_CV_macro_empresa', np.nan)) if resultados[t][alg][1].get('SMAPE_CV_macro_empresa') is not None else None,
                'test_smape_macro_empresa': float(metricas_teste[t][alg].get('SMAPE_macro_empresa', np.nan)) if metricas_teste[t][alg].get('SMAPE_macro_empresa') is not None else None,
                'test_theilu_macro_empresa': float(metricas_teste[t][alg].get('TheilU_macro_empresa', np.nan)) if metricas_teste[t][alg].get('TheilU_macro_empresa') is not None else None,
            }
            for alg in resultados[t].keys()
        }
        for t in resultados.keys()
    }
}
with open(PASTA_SAIDA / 'logs' /'relatorio_modelagem_v2.json', 'w', encoding='utf-8') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)

print('\n' + '═' * 90)
print('RESUMO FINAL — Script 3 Company-Aware')
print('═' * 90)
print(f'Treino: {len(treino):,} obs | Teste: {len(teste):,} obs')
print(f'Features originais: {len(FEATURES)}')
print(f'Modelos treinados: {len(TARGETS) * len(ALGORITMOS)}')
print(f'CV: Walk-Forward {N_SPLITS_WF} folds')
print(f'Pesos amostrais: empresa + futuro repetido')
print(f'Flag COVID: {sorted(COVID_ANOS)}')
print('Artefatos salvos em outputs/')
print('  - modelo_<TARGET>_<ALG>.pkl')
print('  - resultados_cv.csv / resultados_teste.csv')
print('  - resultados_cv.pkl / metricas_teste.pkl / baselines.pkl')
print('  - feature_importances.pkl / melhores_modelos.pkl')
print('  - selected_features_por_target.pkl')
print('  - feature_importance.png / analise_residuos.png')
print('  - predicoes_teste_detalhadas.parquet / .csv')
print('  - relatorio_modelagem.json')
print('═' * 90)
print('✅ Pronto para o Script 4 (Avaliação + Z\'\' )')
print('═' * 90)

print('Artefatos salvos em outputs/')
print('  - modelos individuais por target/algoritmo')
print('  - resultados_cv.csv / resultados_teste.csv')
print('  - metricas_teste.pkl / baselines.pkl / melhores_modelos.pkl')
print('  - feature_importance.png / analise_residuos.png')
print('  - predicoes_teste_detalhadas.parquet / .csv')
print('  - relatorio_modelagem.json')
print('═' * 90)
print('✅ Pronto para o Script 4 (Avaliação + Z\'\' )')
print('═' * 90)
